In [ ]:
# ============================================================
# 🚀 MBERT Multilingual NER Training (AI4Privacy 400k)
# ------------------------------------------------------------
# - Uses ai4privacy/pii-masking-400k
# - Uses local model folder: ./mbert/
# - Uses provided mbert_tokens + BIO labels
# - GPU optimized (RTX A1000 6GB)
# - Compatible with Transformers 2.x (eval_strategy)
# ============================================================

%pip install transformers datasets seqeval accelerate -q

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# 0️⃣ VERIFY GPU
# ------------------------------------------------------------

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 1️⃣ LOAD AI4PRIVACY DATASET
# ------------------------------------------------------------

dataset = load_dataset("ai4privacy/pii-masking-400k")

train_data = dataset["train"]
val_data = dataset["validation"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))


# ------------------------------------------------------------
# 2️⃣ BUILD LABEL → ID MAPPING
# ------------------------------------------------------------

all_labels = set()

for example in train_data:
    for label in example["mbert_token_classes"]:
        all_labels.add(label)

label_list = sorted(list(all_labels))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

num_labels = len(label_list)

print("Number of labels:", num_labels)


# ------------------------------------------------------------
# 3️⃣ LOAD LOCAL MBERT TOKENIZER & MODEL
# ------------------------------------------------------------

tokenizer = BertTokenizerFast.from_pretrained(
    "bert-base-multilingual-cased"
)

model = BertForTokenClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


# ------------------------------------------------------------
# 4️⃣ ENCODE DATA (ALREADY TOKENIZED FOR MBERT)
# ------------------------------------------------------------

def encode_example(example):
    encoding = tokenizer(
        example["mbert_tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        padding=False,
    )

    word_ids = encoding.word_ids()

    labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            label = example["mbert_token_classes"][word_idx]
            labels.append(label2id[label])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    encoding["labels"] = labels
    return encoding


tokenized_train = train_data.map(encode_example)
tokenized_val = val_data.map(encode_example)

data_collator = DataCollatorForTokenClassification(tokenizer)


# ------------------------------------------------------------
# 5️⃣ METRICS
# ------------------------------------------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_seq.append(id2label[p])
                l_seq.append(id2label[l])
        true_preds.append(p_seq)
        true_labels.append(l_seq)

    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


# ------------------------------------------------------------
# 6️⃣ TRAINING CONFIG (GPU OPTIMIZED)
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./ml-ner-model",

    learning_rate=3e-5,
    per_device_train_batch_size=8,      # safe for 6GB
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,      # effective batch size = 16
    num_train_epochs=3,

    weight_decay=0.01,

    fp16=True,                          # GPU acceleration
    logging_steps=500,

    eval_strategy="epoch",
    save_strategy="epoch",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


# ------------------------------------------------------------
# 7️⃣ TRAIN
# ------------------------------------------------------------

trainer.train()


# ------------------------------------------------------------
# 8️⃣ FINAL EVALUATION
# ------------------------------------------------------------

metrics = trainer.evaluate()
print("Final evaluation:", metrics)


# ------------------------------------------------------------
# 9️⃣ SAVE MODEL
# ------------------------------------------------------------

model.save_pretrained("ml-ner-model")
tokenizer.save_pretrained("ml-ner-model")

print("🎉 Training complete! Model saved to ml-ner-model/")

Note: you may need to restart the kernel to use updated packages.


c:\Users\andreea.asimine\OneDrive - Siemens Energy\Desktop\UserPrivacy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Using GPU: NVIDIA RTX A1000 6GB Laptop GPU
Train size: 325517
Validation size: 81379
Number of labels: 35


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1109.11it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]             
BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEX

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.167340,0.065355,0.973497,0.589760
2,0.100481,0.044251,0.982592,0.756737
3,0.060377,0.032861,0.988168,0.851202


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]


Final evaluation: {'eval_loss': 0.032860759645700455, 'eval_accuracy': 0.9881682828247447, 'eval_f1': 0.8512016496416625, 'eval_runtime': 9334.5414, 'eval_samples_per_second': 8.718, 'eval_steps_per_second': 1.09, 'epoch': 3.0}


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.48s/it]

🎉 Training complete! Model saved to ml-ner-model/


In [18]:
# ============================================================
# FULL EVALUATION CELL — CHECKPOINT 61035
# ============================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# 1️⃣ Load Dataset
# ------------------------------------------------------------
print("Loading dataset...")
dataset = load_dataset("ai4privacy/pii-masking-400k")

train_data = dataset["train"]
val_data = dataset["validation"]

print("Validation size:", len(val_data))


# ------------------------------------------------------------
# 2️⃣ Rebuild Label Mapping (must match training)
# ------------------------------------------------------------
print("Rebuilding label mappings...")

all_labels = set()
for example in train_data:
    for label in example["mbert_token_classes"]:
        all_labels.add(label)

label_list = sorted(list(all_labels))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print("Number of labels:", len(label_list))


# ------------------------------------------------------------
# 3️⃣ Load Model Checkpoint 61035
# ------------------------------------------------------------
MODEL_PATH = Path.cwd() / "ml-ner-model" / "checkpoint-61035"

print("Loading model from:", MODEL_PATH)
print("Exists:", MODEL_PATH.exists())

model = BertForTokenClassification.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True
)

print("Classifier weight mean:",
      model.classifier.weight.abs().mean().item())


# ------------------------------------------------------------
# 4️⃣ Load Tokenizer
# ------------------------------------------------------------
tokenizer = BertTokenizerFast.from_pretrained(
    "bert-base-multilingual-cased"
)


# ------------------------------------------------------------
# 5️⃣ Encode Validation Data
# ------------------------------------------------------------
def encode_example(example):
    encoding = tokenizer(
        example["mbert_tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        padding=False,
    )

    word_ids = encoding.word_ids()
    labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            label = example["mbert_token_classes"][word_idx]
            labels.append(label2id[label])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    encoding["labels"] = labels
    return encoding


print("Encoding validation set...")
tokenized_val = val_data.map(encode_example)


# ------------------------------------------------------------
# 6️⃣ Metrics Function
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_seq.append(id2label[p])
                l_seq.append(id2label[l])
        true_preds.append(p_seq)
        true_labels.append(l_seq)

    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


# ------------------------------------------------------------
# 7️⃣ Create Trainer (Evaluation Only)
# ------------------------------------------------------------
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


# ------------------------------------------------------------
# 8️⃣ Evaluate
# ------------------------------------------------------------
print("\nRunning evaluation...")
metrics = trainer.evaluate()


# ------------------------------------------------------------
# 9️⃣ Display Results Cleanly
# ------------------------------------------------------------
display_df = pd.DataFrame({
    "Metric": ["Loss", "Accuracy", "F1 Score", "Runtime (sec)"],
    "Value": [
        f"{metrics['eval_loss']:.6f}",
        f"{metrics['eval_accuracy']:.4f}",
        f"{metrics['eval_f1']:.4f}",
        f"{metrics['eval_runtime']:.2f}",
    ]
})

print("\n===== FINAL EVALUATION (Checkpoint 61035) =====")
display(display_df)

Loading dataset...
Validation size: 81379
Rebuilding label mappings...
Number of labels: 35
Loading model from: c:\Users\andreea.asimine\OneDrive - Siemens Energy\Desktop\UserPrivacy\bert\ml-ner-model\checkpoint-61035
Exists: True


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1113.45it/s, Materializing param=classifier.weight]                                     


Classifier weight mean: 0.021435189992189407
Encoding validation set...


Map: 100%|██████████| 81379/81379 [00:30<00:00, 2698.20 examples/s]



Running evaluation...


KeyboardInterrupt: 

In [16]:
from transformers import Trainer, DataCollatorForTokenClassification

# Make sure tokenized_val and compute_metrics already exist
# If not, re-run the dataset + encode cells.

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

NameError: name 'tokenized_val' is not defined

In [15]:
metrics = trainer.evaluate()

print("\n===== FINAL EVALUATION (Checkpoint 61035) =====")
print(f"Loss:      {metrics['eval_loss']:.6f}")
print(f"Accuracy:  {metrics['eval_accuracy']:.4f}")
print(f"F1 Score:  {metrics['eval_f1']:.4f}")
print(f"Runtime:   {metrics['eval_runtime']:.2f} sec")
print("===============================================")

NameError: name 'trainer' is not defined